In [1]:
import xml.etree.ElementTree as ET
from collections import Counter
import re

def check_page_numbers(root, namespace):
    """
    Finds and reports missing and duplicate page numbers from <pb> tags.
    """
    print("## Page Number (<pb n='N'/>) Analysis ##")
    page_numbers = []
    # Find all <pb> tags and extract the 'n' attribute
    for pb_element in root.findall('.//tei:pb', namespace):
        n_attr = pb_element.get('n')
        if n_attr and n_attr.isdigit():
            page_numbers.append(int(n_attr))

    if not page_numbers:
        print("No valid <pb n='N'/> tags with numeric values found.")
        return

    page_numbers.sort()
    min_page = page_numbers[0]
    max_page = page_numbers[-1]

    # --- Find Missing Pages ---
    full_range = set(range(min_page, max_page + 1))
    present_pages = set(page_numbers)
    missing_pages = sorted(list(full_range - present_pages))

    if missing_pages:
        print(f"\nMissing Page Numbers (from {min_page} to {max_page}):")
        print(", ".join(map(str, missing_pages)))
    else:
        print("\nNo missing page numbers found in the sequence.")

    # --- Find Duplicate Pages ---
    page_counts = Counter(page_numbers)
    # Most page numbers appear twice (Latin/English). Report pages that appear more than twice.
    duplicates = {page: count for page, count in page_counts.items() if count > 2}

    if duplicates:
        print("\nUnusual Duplicate Page Numbers Found (more than 2 occurrences):")
        for page, count in duplicates.items():
            print(f"  - Page '{page}' appears {count} times.")
    else:
        print("\nNo unusual duplicate page numbers found.")

def check_ref_note_correspondence(root, namespace):
    """
    Checks for a one-to-one relationship between <ref> and <note> tags
    based on their 'n' attribute.
    """
    print("## Reference vs. Note (<ref n='PG.N'/> vs. <note n='PG.N'/>) Analysis ##")
    # Regex to match the PG.N format
    ref_pattern = re.compile(r'^\d+\.\d+$')

    # --- Extract 'n' attributes from <ref> and <note> tags ---
    ref_ids = set()
    for ref in root.findall('.//tei:ref', namespace):
        n_attr = ref.get('n')
        if n_attr and ref_pattern.match(n_attr):
            ref_ids.add(n_attr)

    note_ids = set()
    for note in root.findall('.//tei:note', namespace):
        n_attr = note.get('n')
        if n_attr and ref_pattern.match(n_attr):
            note_ids.add(n_attr)

    if not ref_ids and not note_ids:
        print("No <ref> or <note> tags with the format 'PG.N' found.")
        return

    # --- Compare the sets to find mismatches ---
    refs_without_notes = sorted(list(ref_ids - note_ids))
    notes_without_refs = sorted(list(note_ids - ref_ids))

    if not refs_without_notes and not notes_without_refs:
        print("\nSuccess: A one-to-one relationship exists between all <ref> and <note> tags.")
        print(f"Found {len(ref_ids)} matching pairs.")
    else:
        print("\nMismatch Found:")
        if refs_without_notes:
            print("\n  - References <ref> without a matching <note>:")
            print("    " + ", ".join(refs_without_notes))
        if notes_without_refs:
            print("\n  - Notes <note> without a matching <ref>:")
            print("    " + ", ".join(notes_without_refs))

def check_xml_file(file_name):
    """
    Analyzes an XML file to check for missing/duplicate page numbers
    and mismatches between <ref> and <note> tags.

    Args:
        file_name (str): The path to the XML file to be checked.
    """
    try:
        # Register the TEI namespace to properly find elements
        tree = ET.parse(file_name)
        root = tree.getroot()
        namespace = {'tei': 'http://www.tei-c.org/ns/1.0'}

        print(f"--- Analysis Report for: {file_name} ---\n")

        # 1. Check for missing and duplicate page numbers (<pb n="N"/>)
        check_page_numbers(root, namespace)

        print("\n" + "="*40 + "\n")

        # 2. Check for one-to-one relationship between <ref n="PG.N"/> and <note n="PG.N">
        check_ref_note_correspondence(root, namespace)

    except FileNotFoundError:
        print(f"Error: The file '{file_name}' was not found.")
        print("Please make sure the XML file is in the same directory as your notebook, or provide the full path.")
    except ET.ParseError as e:
        print(f"Error: Failed to parse the XML file. Details: {e}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")




In [11]:
# --- EXECUTION CELL ---
# To run this in a Jupyter Notebook, place your XML file in the same
# directory or provide the full path to the file.
# Then, simply run this cell.

# Replace with the name of your file
file_to_check = '/Users/gcrane/github/GRC_misc/phi0474.phil049.king1.xml'

# Call the main function to perform the analysis
check_xml_file(file_to_check)

--- Analysis Report for: /Users/gcrane/github/GRC_misc/phi0474.phil049.king1.xml ---

## Page Number (<pb n='N'/>) Analysis ##

Missing Page Numbers (from 2 to 547):
397

No unusual duplicate page numbers found.


## Reference vs. Note (<ref n='PG.N'/> vs. <note n='PG.N'/>) Analysis ##

Mismatch Found:

  - Notes <note> without a matching <ref>:
    514.1, 514.2


In [18]:
def roman_to_int(roman: str) -> int:
    """
    Convert a Roman numeral (up to C = 100) into an integer.
    Handles subtractive notation (IV = 4, IX = 9, XL = 40, XC = 90).
    """
    roman = roman.upper()
    values = {
        "I": 1,
        "V": 5,
        "X": 10,
        "L": 50,
        "C": 100
    }

    total = 0
    i = 0
    while i < len(roman):
        # If this character is followed by a larger one, subtract it
        if i + 1 < len(roman) and values[roman[i]] < values[roman[i + 1]]:
            total += values[roman[i + 1]] - values[roman[i]]
            i += 2
        else:
            total += values[roman[i]]
            i += 1
    return total


englishfname = re.sub('king1','king1_english',file_to_check)

f = open(re.sub('king1','king1_english',file_to_check))
orgpages = {}
transpages = {}
oldtranspage = 1
oldorgpage = 0

prevchapter = 0

textfname = re.sub('king1','kingtxt1',file_to_check)
notefname = re.sub('king1','kingnote1',file_to_check)

#noteoutf = open(notefname,'w')
#textoutf = open(textfname,'w')
print(textfname,notefname)

def addentry(curdict,curitem):
    if(curitem in curdict):
        curdict[curitem] = curdict[curitem] + 1
    else:
        curdict[curitem] = 1
    
notedict = {}
refdict = {}
curpage = 'NA'
for l in f:
    m = re.search('<p>([IXVLC]+)\.\s+',l)
    if(m):
        subval = str(roman_to_int(m[1]))
        l = re.sub('<p>([IXVLC]+)\.\s+','<p><milestone unit="chapter" n="'+str(roman_to_int(m[1]))+'"/>',l)
        print(curpage,roman_to_int(m[1]))
    m = re.search('<pb n="([0-9]*[13579])"',l)
    if(m):
        curpage = m[1]
        newtranspage = int(m[1])
        if(not newtranspage == oldtranspage + 2):
            print('trans',oldtranspage,newtranspage)
        oldtranspage = newtranspage

    m = re.search('<pb n="([0-9]*[02468])"',l)
    if(m):
        neworgpage = int(m[1])
        curpage = m[1]
        if(not neworgpage == oldorgpage + 2):
            print('trans',oldorgpage,neworgpage)
        oldorgpage = neworgpage

    m = re.search('<milestone unit="chapter" n="([[0-9]+)"',l)
    if(m):
        curchapter = int(m[1])
#        if(not curchapter == prevchapter + 1):
#            print('jump',curpage,prevchapter,curchapter,l)
        prevchapter = curchapter
    
    savel = l
    while(re.search('<note n="([^"]+)"',savel)):
        m = re.search('<note n="([^"]+)"',savel)
        if(m):
            addentry(notedict,m[1])
        savel = re.sub('<note n="([^"]+)"','',savel,1)

    savel = l
    while(re.search('<ref n="([^"]+)"',savel)):
        m = re.search('<ref n="([^"]+)"',savel)
        if(m):
            addentry(refdict,m[1])
        savel = re.sub('<ref n="([^"]+)"','',savel,1)

f.close()

for foo in refdict:
    if(not foo in notedict):
        print('note missing',foo)
    if(not refdict[foo] == 1):
        print('refover',foo,refdict[foo])
        
for foo in notedict:
    if(not foo in refdict):
        print('ref missing',foo)
    if(not notedict[foo] == 1):
        print('noteover',foo,notedict[foo])

/Users/gcrane/github/GRC_misc/phi0474.phil049.kingtxt1.xml /Users/gcrane/github/GRC_misc/phi0474.phil049.kingnote1.xml
107 37
123 43
125 44
129 45
135 47
137 48
147 1
151 2
153 3
155 4
157 5
159 5
161 6
167 8
175 12
177 13
181 14
183 15
185 16
189 17
193 18
199 20
201 21
205 22
209 23
211 24
213 25
217 26
219 27
225 1
227 2
229 3
231 4
235 5
239 6
241 7
243 8
247 9
251 10
255 11
257 12
259 13
261 14
263 15
267 16
269 17
337 5
389 25
425 1
433 4
455 10
463 13
465 14
ref missing GRC
ref missing GRC: Another point:
